In [ ]:
import sys
import os
import subprocess

import matplotlib.pyplot as plt
import matplotlib as mpl

%matplotlib inline
params_large = {'legend.fontsize': 'medium',
                'figure.figsize': (10,6),
                'axes.labelsize': 'large',
                'axes.titlesize':'large',
                'xtick.labelsize':'large',
                'ytick.labelsize':'large'}
params_medium = {'legend.fontsize': 'medium',
                'figure.figsize': (10,6),
                'axes.labelsize': 'medium',
                'axes.titlesize':'medium',
                'xtick.labelsize':'medium',
                'ytick.labelsize':'medium'}
params_small = {'legend.fontsize': 'small',
                'figure.figsize': (10,6),
                'axes.labelsize': 'small',
                'axes.titlesize':'medium',
                'xtick.labelsize':'small',
                'ytick.labelsize':'small'}
plt.rcParams.update(params_large)
# mpl.rcParams['xtick.labelsize'] = 10

import src.plot_fns as pltg             # generic plot fns
import src.plot_h5_psd_sg1 as plt_h5    # blimpy-based plot fns

import math
import numpy as np
from astropy import units as u
from astropy.coordinates import Angle
import blimpy as bl
import time
import pandas
import h5py

from pathlib import Path

sys.path.append(os.getenv('SETIGEN_PATH'))

import setigen as stg

try:
    sg_dir
except NameError:
    sg_dir = os.getenv('SGDIR') + '/'
    
if not os.path.isdir(sg_dir[0:-1]):
    os.system('mkdir '+sg_dir[0:-1])

output_dir = os.getenv('OUTDIR') + '/'
if not os.path.isdir(output_dir[0:-1]):
    os.system('mkdir '+output_dir[0:-1])

def db(x):
    """ Convert linear value to dB value """
    return 10*np.log10(np.abs(x.astype(np.float64))+1e-20)

def find(cond,dim=0):
    """ Return indices according to conditon, e.g. find(x>2), like matlab find() """
    return np.nonzero(cond)[dim]

def in_interval(f,f12):
    """ Return True if f within f12 = [f1,f2] interval, e.g. in_interval(f,[f1,f2]) """
    f1 = f12[0]
    f2 = f12[1]
    if (f1<f2):
        return (f>=f1)&(f<=f2)
    else:
        return (f<=f1)&(f>=f2)
    

In [ ]:
n_coarse_chnl = 8
output_file_name_base = 'synthetic_chi2'
if (1):
    pfb_file_name = 'GBT_spliced_PFB_response.f32'
    pfb_string = 'pfb0'
elif (1):
    pfb_file_name = 'chnl_1024K_8_64ch.f32'
    pfb_string = 'pfb8'
elif (1):
    pfb_file_name = 'chnl_1024K_16_64ch.f32'
    pfb_string = 'pfb16'
elif (1):
    pfb_file_name = 'chnl_1024K_12_64ch.f32'
    pfb_string = 'pfb12'

# define a linear slope to the channels, so that the noise amplitude varies over total_slope_delta_db decibels left to right
if (0):
    total_slope_delta_db = 0.
elif (1):
    total_slope_delta_db = 2.

pfb_string += f'_{total_slope_delta_db:.0f}dB'

In [ ]:
### Code reference: Ben Jacobsen-Bell setigen_fully_synthetic.ipynb
### Load in single-channel Voyager 1 data to compute some GBT noise statistics.
### If you're using a different telescope, you may wish to load a different HDF5 file for waterfall_fn.

waterfall_fn = sg_dir + 'single_coarse_guppi_59046_80036_DIAG_VOYAGER-1_0011.rawspec.0000.h5'

fb = bl.Waterfall(waterfall_fn)
print(fb.header)

### Perform a sigma-clipping routine to remove the Voyager signal and most of the PFB rolloff.
### (See C. Choza et al 2024 for justification.)
from astropy.stats import sigma_clip
clipped_data = sigma_clip(fb.data,
                                  sigma=5,
                                  maxiters=5,
                                  masked=False)
### Compute the noise floor and rms. We will use the noise floor to initialize our synthetic frame below.
print(fb.data.shape)
n_sti = np.round(fb.header['tsamp']*abs(fb.header['foff']*1e6)).astype(int)
n_lti = fb.data.shape[0]
n_avg = n_sti*n_lti
n_fft = fb.data.shape[2]
print(f'{n_sti=}, {n_lti=}, {n_avg=}, {n_fft=}')
print(clipped_data.shape)
noise_mean = np.mean(clipped_data)
noise_std = np.std(clipped_data)
std_mean_ratio_obs = noise_std/noise_mean/np.sqrt(n_lti)
std_mean_ratio_nom = 1./np.sqrt(2.*n_avg)
obs_std_mean_excess = std_mean_ratio_obs/std_mean_ratio_nom
print(f'{noise_mean=}')
print(f'{noise_std=}')
print(f'clipped noise_std/noise_mean/sqrt(n_lti)={std_mean_ratio_obs:.4f} vs. 1/sqrt(2*n_avg)={std_mean_ratio_nom:.4f}')
print(f'{obs_std_mean_excess=:.2f}')

del clipped_data, fb



In [ ]:
### We need our synthetic data to be readable by turboSETI/BLISS, so we will make it into an HDF5 file with BL's usual header style.
### Most of this information is a placeholder with the precise value being unimportant.

head = {'DIMENSION_LABELS': np.array([b'time', b'feed_id', b'frequency'], dtype=object), 
        'az_start': 0.0, 
        'data_type': 1, 
        'fch1': 6095.214842353016, # This is the edge of a C-band node; it can be changed, but other values below will then also have to be changed.
        'foff': -2.7939677238464355e-06, # Fine-channel frequency resolution.
        'machine_id': 20, 
        'nbits': 32, 
        'nchans': 1048576*n_coarse_chnl, # n_lti coarse channels of 2e20 fine channels each.
        'nifs': 1, 
        'source_name': 'synthetic', # Placeholder name. Some Blimpy functions care about the source name.
        'src_dej': Angle(0*u.deg), # Placeholder declination.
        'src_raj': Angle('0h0m0s'), # Placeholder RA.
        'telescope_id': 6, 
        'tsamp': 18.253611008, # Time resolution.
        'tstart': 60000, # Arbitrary epoch of observation.
        'za_start': 0.0}

In [ ]:
### In this cell, we generate n_coarse_chnl coarse channels' worth of chi2 noise using the noise mean we computed from the real GBT data above.

### Read in the polyphase filterbank (PFB) shape. 
### If you're using a different telescope, you can generate a custom PFB shape using BLISS.
### Type the following command in the terminal on the Berkeley Data Center to see how:
### bliss_generate_channelizer_response -h
long_data = []
pfb = np.fromfile(pfb_file_name, dtype='float32')

slope_mult2 = 10**(.1*total_slope_delta_db/2.*np.sign(head['foff']))
slope_mult1 = 1. - (slope_mult2 - 1.)
slope_mult_coarse = np.linspace(slope_mult1,slope_mult2,n_coarse_chnl+1,True)
print(slope_mult_coarse)

### Do one coarse channel at a time.
### Setigen frame construction has a nonlinear complexity, so this is faster than doing a single frame with n_coarse_chnl channels.
for i in range(n_coarse_chnl):

    print(i)

    fchans = (2**20)*1
    tchans = n_lti
    df = 2.7939677238464355*u.Hz
    dt = 18.253611008*u.s
    fch1 = 6095.214842353016*u.MHz

    if (0):
        frame = stg.Frame(fchans=fchans,
                    tchans=tchans,
                    df=df,
                    dt=dt,
                    fch1=fch1)
    else:
        frame = stg.Frame.from_data(df=df,
                dt=dt,
                fch1=fch1,
                ascending=False,
                data=np.zeros((tchans,fchans),dtype='float32'))
        
    noise = frame.add_noise(x_mean=noise_mean)

    # print(frame.data.shape)
    # print(pfb.shape)
    # print((frame.data*pfb).shape)
    # print(frame.data*pfb)
    
    slope_fine = np.linspace(slope_mult_coarse[i],slope_mult_coarse[i+1],n_fft,False)
    print(slope_fine.shape)
    print('min=',min(slope_fine),' max=',max(slope_fine))

    pfb_data = frame.data*(pfb*slope_fine)
    # pfb_data = np.array([spec*pfb for spec in frame.data])
    print(pfb_data.shape)
    # print(pfb_data)
    
    long_data.append(pfb_data)

del pfb_data

In [ ]:
print(len(long_data))
print(long_data[0].shape)

In [ ]:
print(noise.shape)
noise_mean = np.mean(noise)
noise_std = np.std(noise)
print(f'{noise_mean=}')
print(f'{noise_std=}')
std_mean_ratio_chi2 = noise_std/noise_mean/np.sqrt(n_lti)
std_mean_ratio_nom = 1./np.sqrt(2.*n_avg)
chi2_std_mean_excess = std_mean_ratio_chi2/std_mean_ratio_nom
print(f'chi2 noise_std/noise_mean/sqrt(n_lti)={std_mean_ratio_chi2:.4f} vs. 1/sqrt(2*n_avg)={std_mean_ratio_nom:.4f}')
print(f'{chi2_std_mean_excess=:.2f}')


In [ ]:
whos

In [ ]:
### Stack your n_coarse_chnl coarse channels together into a single array.

concat_long_data = np.hstack(long_data)

del long_data

In [ ]:
concat_long_data.shape

In [ ]:
### Convert your N-channel numpy array back to a setigen Frame.
### Remember to use the metadata we defined earlier!

frame = stg.Frame.from_data(df = 2.7939677238464355*u.Hz,
                            dt = 18.253611008*u.s,
                            fch1=6095.214842353016*u.MHz,
                            ascending=False,
                            data=concat_long_data,
                            metadata=head)
del concat_long_data

In [ ]:
### Save the n_coarse_chnl channels of noise as an HDF5 file.

fil_file_name = sg_dir + output_file_name_base + f'_{pfb_string}' + f'_{n_coarse_chnl}ch.fil'
h5_file_name  = sg_dir + output_file_name_base + f'_{pfb_string}' + f'_{n_coarse_chnl}ch.h5'

# frame.save_h5(h5_file_name)

frame.save_fil(fil_file_name)
# from blimpy import Waterfall
# wf = Waterfall(fil_file_name)
# wf.write_to_hdf5(h5_file_name)

In [ ]:
### Read the noise file back in as a Waterfall object to check that everything looks right.

fb = bl.Waterfall(fil_file_name)
fb.info()

In [ ]:
if (1):
    try:
        # Beep in WSL
        if os.system("powershell.exe '[console]::beep(261.6,700)'") !=0:
            raise Exception('powershell.exe not found')
    except:
        # linux, probably doesn't work
        print('Beep!')
        os.system("echo -ne '\a'")